# **[실습 3] 손그림 업그레이드**

## **[3-1] 프롬프트 엔지니어링과 OpenAI API 모델 활용하여 이미지 변환하기**

In [2]:
import os
import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI
import base64
import numpy as np
from PIL import Image
import io

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

client = OpenAI(api_key=api_key)

In [8]:
import io
import base64

import gradio as gr
from PIL import Image


def draw(image, user_prompt):
    # 1. 그림판에서 배경과 그리기 레이어가 합쳐진 이미지 가져오기
    if image is None or image.get("composite") is None:
        raise gr.Error("왼쪽 그림판에 먼저 그림을 그려주세요.")

    sketch = image["composite"].convert("RGBA")

    # 2. 투명한 부분을 흰색 배경으로 채우기
    background = Image.new("RGBA", sketch.size, "white")
    sketch = Image.alpha_composite(background, sketch).convert("RGB")

    # 3. PIL 이미지 -> PNG 파일 객체
    buffer = io.BytesIO()
    sketch.save(buffer, format="PNG")
    buffer.seek(0)

    # 4. 역할 설명과 변환 요청을 하나의 프롬프트로 구성
    prompt = f"""
    너는 세계적인 애니메이션 게임 캐릭터 디자이너야!.

    입력 이미지는 사용자가 직접 그린 스케치야.
    스케치의 주요 대상, 구도, 배치를 유지하면서
    정교한 명암, 풍부한 색감, 섬세한 질감이 있는
    마블에 있는 캐릭터 처럼 완성해줘.

    사용자의 추가 요청:
    {user_prompt}
    """

    # 5. 그림판 이미지를 모델에 전달
    response = client.images.edit(
        model="gpt-image-1-mini",
        image=("sketch.png", buffer.getvalue(), "image/png"),
        prompt=prompt,
        size="1024x1024",
        n=1,
    )

    # 6. 생성된 base64 이미지 -> PIL 이미지
    if not response.data or not response.data[0].b64_json:
        raise gr.Error("이미지가 반환되지 않았습니다. 다시 시도해주세요.")

    output_bytes = base64.b64decode(response.data[0].b64_json)

    return Image.open(io.BytesIO(output_bytes)).copy()


with gr.Blocks() as demo:
    gr.Markdown("## 🎨 내가 그린 그림을 원하는대로 바꾸기")

    with gr.Row():
        # 왼쪽: 직접 그릴 수 있는 그림판
        with gr.Column():
            input_img = gr.Sketchpad(
                label="여기에 그림을 그려주세요",
                type="pil",
                image_mode="RGBA",
                format = "png",
                canvas_size=(800, 800),
                height=500,
                brush=gr.Brush(
                    colors=[
                        "#000000",
                        "#FF0000",
                        "#0000FF",
                        "#008000",
                        "#FFFF00",
                    ],
                    color_mode="defaults",
                ),
            )

            prompt_box = gr.Textbox(
                label="어떻게 바꿀까요?",
                value="예술적 완성도가 높은 르네상스 명화로 바꿔줘.",
                placeholder="예: 따뜻한 석양 아래 풍경을 유화로 그려줘.",
                lines=3,
            )

            btn = gr.Button("명작화", variant="primary")

        # 오른쪽: 변환 결과
        with gr.Column():
            output_img = gr.Image(
                label="완성된 그림",
                type="pil",
                height=500,
                interactive=False,
            )

    btn.click(
        fn=draw,
        inputs=[input_img, prompt_box],
        outputs=output_img,
        concurrency_limit=1,
    )

demo.queue().launch()

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


## **[3-2] Gradio UI 접목하기**

In [2]:
# code here